In [2]:
import torch

torch.manual_seed(42)   # 固定随机种子，保证结果可复现

# ============ 1. 构造输入 ============
B, H, n, d_head = 2, 4, 6, 8     # batch=2, 头数=4, 序列长=6, 每头维度=8

# 随机生成 q, k（模拟一个注意力层内部的投影结果）
q = torch.randn(B, H, n, d_head)
k = torch.randn(B, H, n, d_head)

# ============ 2. 缩放点积注意力分数 ============
# scores[b, h, i, j] = q_i · k_j / sqrt(d_head)
scores = q @ k.transpose(-2, -1) / (d_head ** 0.5)   # 形状: (B, H, n, n)
print("scores 形状:", scores.shape)                    # torch.Size([2, 4, 6, 6])

# ============ 3. softmax 得到行随机权重矩阵 ============
attn = torch.softmax(scores, dim=-1)                  # (B, H, n, n)
print("attn 形状:", attn.shape)                        # torch.Size([2, 4, 6, 6])
print("每行和 = 1 验证:", attn.sum(dim=-1)[0, 0])       # 全为 1.0

# ============ 4. 有效邻域大小（Effective Sample Size） ============
# 对每个 query 位置 i：ess_i = 1 / sum_j attn[b,h,i,j]^2
ess = 1.0 / attn.pow(2).sum(dim=-1)                   # (B, H, n)
print("ess 形状:", ess.shape)                          # torch.Size([2, 4, 6])
print("ess 取值范围验证: min = %.4f, max = %.4f" % (ess.min(), ess.max()))

# ============ 5. 两个极端情况理解 ess ============
# (a) 均匀分布（注意力完全分散）: ess → n
attn_uniform = torch.full((1, 1, 1, n), 1.0 / n)
ess_uniform = 1.0 / attn_uniform.pow(2).sum(dim=-1)
print(f"\n均匀分布时 ess = {ess_uniform.item():.1f}  (等于 n = {n})")

# (b) one-hot（注意力完全集中）: ess = 1
attn_onehot = torch.zeros(1, 1, 1, n)
attn_onehot[..., 0] = 1.0
ess_onehot = 1.0 / attn_onehot.sqrt().pow(2).sum(dim=-1)
print(f"one-hot 时   ess = {ess_onehot.item():.1f}  (等于 1)")

# ============ 6. 打印部分结果 ============
print("\n第 1 个样本、第 1 个头的注意力矩阵 (6x6):")
print(attn[0, 0].round(decimals=3))

print("\n对应的 ess（每个 query 的有效邻居数）:")
print(ess[0, 0].round(decimals=3))


scores 形状: torch.Size([2, 4, 6, 6])
attn 形状: torch.Size([2, 4, 6, 6])
每行和 = 1 验证: tensor([1.0000, 1.0000, 1.0000, 1.0000, 1.0000, 1.0000])
ess 形状: torch.Size([2, 4, 6])
ess 取值范围验证: min = 1.5638, max = 5.6464

均匀分布时 ess = 6.0  (等于 n = 6)
one-hot 时   ess = 1.0  (等于 1)

第 1 个样本、第 1 个头的注意力矩阵 (6x6):
tensor([[0.4820, 0.0100, 0.1420, 0.0570, 0.2150, 0.0940],
        [0.2560, 0.0960, 0.1970, 0.0770, 0.0910, 0.2820],
        [0.0980, 0.1140, 0.1430, 0.4410, 0.1060, 0.0980],
        [0.2270, 0.0670, 0.0730, 0.3840, 0.1040, 0.1440],
        [0.0810, 0.3840, 0.1000, 0.1160, 0.1960, 0.1230],
        [0.1080, 0.1880, 0.2360, 0.0740, 0.2170, 0.1770]])

对应的 ess（每个 query 的有效邻居数）:
tensor([3.2190, 4.8160, 3.8660, 4.1570, 4.3240, 5.3600])


In [3]:
import numpy as np

rng = np.random.default_rng(0)
n, d, dk = 12, 8, 8
X = rng.normal(size=(n, d))
Wq = rng.normal(size=(d, dk)) / np.sqrt(d)
Wk = rng.normal(size=(d, dk)) / np.sqrt(d)
Wv = rng.normal(size=(d, dk)) / np.sqrt(d)

K, V = X @ Wk, X @ Wv
x_new = rng.normal(size=(d,))          # 固定一个查询点（不在序列里的新 token）
q = x_new @ Wq

# softmax 路径
s = (K @ q) / np.sqrt(dk)
w_softmax = np.exp(s - s.max()); w_softmax /= w_softmax.sum()

# 显式 Nadaraya-Watson 路径：分子 sum_j K(x,x_j) v_j，分母 sum_j K(x,x_j)
num = np.exp(s) @ V
den = np.exp(s).sum()
o_nw = num / den
w_nw = np.exp(s) / den

for h in [0.25, 0.5, 1.0, 2.0, 100.0]:   # 带宽 h：s = q.k / (h * sqrt(dk))
    w = np.exp((K @ q) / (h * np.sqrt(dk))); w = w / w.sum()
    ess = 1.0 / np.sum(w ** 2)
    j = int(np.argmax(w))
    print(f"h={h:>6}: top1权重={w.max():.4f}  j*={j:>2}  ESS={ess:6.2f}  "
          f"o[:3]={np.round((w @ V)[:3], 4)}")

w_hard = np.exp((K @ q) / (0.01 * np.sqrt(dk))); w_hard /= w_hard.sum()
w_soft = np.exp((K @ q) / (1e6 * np.sqrt(dk))); w_soft /= w_soft.sum()
j = int(np.argmax((K @ q)))
print("|o(h->0) - V[j*]|   =", np.abs(w_hard @ V - V[j]).max())
print("|o(h->inf) - meanV| =", np.abs(w_soft @ V - V.mean(axis=0)).max())
print("softmax 与 NW 两条路径的权重差 =", np.abs(w_softmax - w_nw).max())

h=  0.25: top1权重=0.8908  j*= 5  ESS=  1.26  o[:3]=[ 1.1663 -1.1662 -1.1519]
h=   0.5: top1权重=0.5465  j*= 5  ESS=  2.98  o[:3]=[ 0.8956 -0.854  -0.6496]
h=   1.0: top1权重=0.2842  j*= 5  ESS=  6.76  o[:3]=[ 0.6546 -0.5739 -0.2831]
h=   2.0: top1权重=0.1699  j*= 5  ESS=  9.85  o[:3]=[ 0.5204 -0.3624 -0.1118]
h= 100.0: top1权重=0.0847  j*= 5  ESS= 12.00  o[:3]=[ 0.3847 -0.0794  0.0625]
|o(h->0) - V[j*]|   = 0.0
|o(h->inf) - meanV| = 8.138509599575627e-07
softmax 与 NW 两条路径的权重差 = 1.3877787807814457e-17


In [4]:
import numpy as np

rng = np.random.default_rng(1)
sigma = 1.0

# 一、方差论证的经验核对：对 q 与 k 共同平均
print("== 方差论证的经验核对：Var(q.k) = dk * sigma^4（对 q 与 k 共同平均）==")
for dk in [8, 64, 512]:
    m = 2000
    dots = np.empty((m, 2000))
    for i in range(m):
        q = rng.normal(size=dk) * sigma
        K = rng.normal(size=(2000, dk)) * sigma
        dots[i] = K @ q
    print(f"dk={dk:>4}: 经验 sd={dots.std():8.4f}   理论 sqrt(dk)*sigma^2="
          f"{np.sqrt(dk) * sigma ** 2:8.4f}")

# 二、温度缩放对注意力熵的影响
n, dk = 256, 64
q = rng.normal(size=dk)
K = rng.normal(size=(n, dk))
dots = K @ q

def entropy(w):
    w = w[w > 1e-300]
    return float(-np.sum(w * np.log(w)))

Hmax = np.log(n)
print("\n== 温度缩放对注意力熵的影响（n=256 个候选 key）==")
for T in [0.1, 0.25, 0.5, 1.0, 2.0, 4.0]:
    s = dots / (np.sqrt(dk) * T)
    w = np.exp(s - s.max()); w = w / w.sum()
    print(f"T={T:>4}: 熵={entropy(w):.3f} ({entropy(w)/Hmax:.3f} of max)  "
          f"ESS={1.0/np.sum(w**2):7.2f}  最大权重={w.max():.4f}")

# 三、不缩放时的饱和效应
print("\n== 不缩放（除数取 1）的饱和效应 ==")
for dk in [8, 64, 512]:
    q = rng.normal(size=dk); K = rng.normal(size=(n, dk))
    dots = K @ q
    for label, sc in [("除以 sqrt(dk)", np.sqrt(dk)), ("不缩放", 1.0)]:
        s = dots / sc
        w = np.exp(s - s.max()); w = w / w.sum()
        print(f"dk={dk:>4} {label}: 熵/Hmax={entropy(w)/np.log(n):.3f}  "
              f"ESS={1.0/np.sum(w**2):6.1f}  最大权重={w.max():.4f}")

== 方差论证的经验核对：Var(q.k) = dk * sigma^4（对 q 与 k 共同平均）==
dk=   8: 经验 sd=  2.8355   理论 sqrt(dk)*sigma^2=  2.8284
dk=  64: 经验 sd=  7.9994   理论 sqrt(dk)*sigma^2=  8.0000
dk= 512: 经验 sd= 22.6396   理论 sqrt(dk)*sigma^2= 22.6274

== 温度缩放对注意力熵的影响（n=256 个候选 key）==
T= 0.1: 熵=0.641 (0.116 of max)  ESS=   1.52  最大权重=0.7906
T=0.25: 熵=1.958 (0.353 of max)  ESS=   4.09  最大权重=0.4172
T= 0.5: 熵=3.902 (0.704 of max)  ESS=  20.36  最大权重=0.1432
T= 1.0: 熵=5.095 (0.919 of max)  ESS= 107.05  最大权重=0.0366
T= 2.0: 熵=5.432 (0.980 of max)  ESS= 204.32  最大权重=0.0134
T= 4.0: 熵=5.517 (0.995 of max)  ESS= 241.96  最大权重=0.0074

== 不缩放（除数取 1）的饱和效应 ==
dk=   8 除以 sqrt(dk): 熵/Hmax=0.924  ESS= 108.9  最大权重=0.0356
dk=   8 不缩放: 熵/Hmax=0.497  ESS=   7.2  最大权重=0.2434
dk=  64 除以 sqrt(dk): 熵/Hmax=0.909  ESS= 102.7  最大权重=0.0321
dk=  64 不缩放: 熵/Hmax=0.253  ESS=   3.0  最大权重=0.4895
dk= 512 除以 sqrt(dk): 熵/Hmax=0.914  ESS= 107.9  最大权重=0.0345
dk= 512 不缩放: 熵/Hmax=0.011  ESS=   1.0  最大权重=0.9885


In [6]:
import math
import torch
import torch.nn as nn
import torch.nn.functional as F


class MultiHeadAttention(nn.Module):
    """支持 MHA / GQA / MQA 的多头注意力；n_kv_heads=1 时退化为 MQA。"""

    def __init__(self, d_model, n_heads, n_kv_heads=None, causal=True):
        super().__init__()
        self.n_heads, self.n_kv_heads = n_heads, (n_kv_heads or n_heads)
        assert d_model % n_heads == 0 and n_heads % self.n_kv_heads == 0
        self.d_head = d_model // n_heads
        self.causal = causal
        self.q_proj = nn.Linear(d_model, n_heads * self.d_head, bias=False)
        self.k_proj = nn.Linear(d_model, self.n_kv_heads * self.d_head, bias=False)
        self.v_proj = nn.Linear(d_model, self.n_kv_heads * self.d_head, bias=False)
        self.o_proj = nn.Linear(n_heads * self.d_head, d_model, bias=False)

    def forward(self, x):
        B, n, _ = x.shape
        H, Hkv, Dh = self.n_heads, self.n_kv_heads, self.d_head
        q = self.q_proj(x).view(B, n, H, Dh).transpose(1, 2)     # (B, H, n, Dh)
        k = self.k_proj(x).view(B, n, Hkv, Dh).transpose(1, 2)   # (B, Hkv, n, Dh)
        v = self.v_proj(x).view(B, n, Hkv, Dh).transpose(1, 2)
        if Hkv != H:                                             # GQA/MQA：把 kv 头广播对齐到 q 头
            rep = H // Hkv
            k = k[:, :, None].expand(B, Hkv, rep, n, Dh).reshape(B, H, n, Dh)
            v = v[:, :, None].expand(B, Hkv, rep, n, Dh).reshape(B, H, n, Dh)
        scores = (q @ k.transpose(-2, -1)) / math.sqrt(Dh)       # (B, H, n, n)
        if self.causal:                                          # 自回归遮蔽：禁止看未来
            mask = torch.ones(n, n, dtype=torch.bool, device=x.device).triu(1)
            scores = scores.masked_fill(mask, torch.finfo(scores.dtype).min)
        attn = F.softmax(scores, dim=-1)                         # 逐行归一化
        out = (attn @ v).transpose(1, 2).contiguous().view(B, n, H * Dh)
        return self.o_proj(out)


# 用法示例
mha = MultiHeadAttention(d_model=1024, n_heads=16, n_kv_heads=4, causal=True)
y = mha(torch.randn(2, 512, 1024))    # (batch=2, seq=512, d_model=1024)
print(y.shape)

torch.Size([2, 512, 1024])


In [10]:
import torch

def precompute_freqs_cis(dim: int, seq_len: int, base: float = 10000.0):
    theta = 1.0 / (base ** (torch.arange(0, dim, 2).float() / dim))
    pos = torch.arange(seq_len).float()
    freqs = torch.outer(pos, theta)
    return torch.polar(torch.ones_like(freqs), freqs)      # (seq_len, dim/2) complex

def apply_rotary_emb(xq, xk, freqs_cis):
    def to_complex(t):
        t = t.float().reshape(*t.shape[:-1], -1, 2)
        real, imag = t.unbind(-1)
        return torch.complex(real, imag)
    xq_c, xk_c = to_complex(xq), to_complex(xk)
    f = freqs_cis.unsqueeze(1)                              # (n, 1, Dh/2)
    out_q = torch.view_as_real(xq_c * f).flatten(-2)
    out_k = torch.view_as_real(xk_c * f).flatten(-2)
    return out_q.type_as(xq), out_k.type_as(xk)

# ---- 实例化 ----
B, n, H, d_head = 2, 6, 4, 8
q = torch.randn(B, n, H, d_head)
k = torch.randn(B, n, H, d_head)

freqs_cis = precompute_freqs_cis(d_head, n)                 # 只需前 n 个位置
q_rot, k_rot = apply_rotary_emb(q, k, freqs_cis)

print(q_rot.shape)   # torch.Size([2, 6, 4, 8])

# ---- 验证 RoPE 的关键性质：内积只依赖相对位置 ----
# 取相对位置差为 2 的两对 (q_i, k_{i+2})，旋转后内积应与绝对位置无关
q_rot_b = q_rot[0, :, 0, :]   # 去掉 batch 和 head 维方便观察
k_rot_b = k_rot[0, :, 0, :]
print((q_rot_b[0] @ k_rot_b[2]).item(), (q_rot_b[3] @ k_rot_b[5]).item())
# 两个值一般不相等（因为 q,k 本身随机不同），但若比较同一对 (q, k) 在不同绝对位置的旋转结果，内积相同


torch.Size([2, 6, 4, 8])
2.3643617630004883 -3.113664150238037


In [11]:
import numpy as np

def apply_rope(x, pos, base=10000.0):
    """x: (dim,)，pos: 标量位置。按相邻两维分组做平面旋转。"""
    dim = x.shape[-1]
    theta = base ** (-np.arange(0, dim, 2) / dim)   # 每对维度的角频率
    ang = pos * theta
    cos, sin = np.cos(ang), np.sin(ang)
    out = np.empty_like(x)
    out[..., 0::2] = x[..., 0::2] * cos - x[..., 1::2] * sin
    out[..., 1::2] = x[..., 0::2] * sin + x[..., 1::2] * cos
    return out

rng = np.random.default_rng(2)
dim = 64

# 一、正交性：范数保持
x = rng.normal(size=dim)
for m in [0, 7, 133, 4096]:
    print(f"pos={m:>5}: |R_m x| = {np.linalg.norm(apply_rope(x, m)):.12f}")

# 二、相对位置不变性：<R_m q, R_n k> 在整体平移下不变
q = rng.normal(size=dim)
k = rng.normal(size=dim)
worst = 0.0
for (m, n) in [(3, 10), (100, 200), (5, 8), (1000, 1500), (7, 899)]:
    a = apply_rope(q, m) @ apply_rope(k, n)
    b = apply_rope(q, m + 500) @ apply_rope(k, n + 500)   # 两个位置同时 +500
    worst = max(worst, abs(a - b))
    print(f"(m,n)=({m:>4},{n:>4})  d={n-m:>4}  <R_m q, R_n k> = {a:>12.8f}  "
          f"平移后 = {b:>12.8f}")
print("所有配对的最大偏差 =", worst)

# 三、同一向量随相对距离的内积（衰减趋势）
v = rng.normal(size=dim)
base = np.linalg.norm(v) ** 2
for dlt in [0, 1, 2, 4, 8, 16, 32, 64, 128]:
    ip = apply_rope(v, 0) @ apply_rope(v, dlt)
    print(f"相对距离 d={dlt:>4}: <R_0 v, R_d v> / |v|^2 = {ip/base:.4f}")

pos=    0: |R_m x| = 7.780780250995
pos=    7: |R_m x| = 7.780780250995
pos=  133: |R_m x| = 7.780780250995
pos= 4096: |R_m x| = 7.780780250995
(m,n)=(   3,  10)  d=   7  <R_m q, R_n k> =  -2.67005614  平移后 =  -2.67005614
(m,n)=( 100, 200)  d= 100  <R_m q, R_n k> =  -5.15552934  平移后 =  -5.15552934
(m,n)=(   5,   8)  d=   3  <R_m q, R_n k> =   5.08542143  平移后 =   5.08542143
(m,n)=(1000,1500)  d= 500  <R_m q, R_n k> =  -3.47358969  平移后 =  -3.47358969
(m,n)=(   7, 899)  d= 892  <R_m q, R_n k> =   4.03308118  平移后 =   4.03308118
所有配对的最大偏差 = 1.1368683772161603e-13
相对距离 d=   0: <R_0 v, R_d v> / |v|^2 = 1.0000
相对距离 d=   1: <R_0 v, R_d v> / |v|^2 = 0.9729
相对距离 d=   2: <R_0 v, R_d v> / |v|^2 = 0.9035
相对距离 d=   4: <R_0 v, R_d v> / |v|^2 = 0.7541
相对距离 d=   8: <R_0 v, R_d v> / |v|^2 = 0.7254
相对距离 d=  16: <R_0 v, R_d v> / |v|^2 = 0.6935
相对距离 d=  32: <R_0 v, R_d v> / |v|^2 = 0.6878
相对距离 d=  64: <R_0 v, R_d v> / |v|^2 = 0.4430
相对距离 d= 128: <R_0 v, R_d v> / |v|^2 = 0.2502


In [12]:
def cache_bytes(L, n_kv, d_head, b):
    return 2 * L * n_kv * d_head * b          # 因子 2 = K 与 V 两份

configs = [
    # 名称, 层数 L, query 头数 H, kv 头数 n_kv, 头维 d_head
    ("Llama-3-8B 量级", 32, 32, 8, 128),
    ("同规模假想 MHA", 32, 32, 32, 128),
    ("同规模假想 MQA", 32, 32, 1, 128),
    ("Llama-3-70B 量级", 80, 64, 8, 128),
]

for name, L, H, Hkv, dh in configs:
    per = cache_bytes(L, Hkv, dh, 2)          # b=2：FP16
    mb_per = per / 1024 ** 2
    gb_32k = per * 32768 / 1024 ** 3
    print(f"{name:<18}{mb_per:>12.3f} MB/token{gb_32k:>9.2f} GB@32K"
          f"  相对MHA={Hkv/H:.2f}x")

per = cache_bytes(32, 8, 128, 1)              # FP8：b=1
print(f"GQA n_kv=8, FP8: {per/1024:.1f} KB/token, 32K 上下文 "
      f"{per*32768/1024**3:.2f} GB")

Llama-3-8B 量级            0.125 MB/token     4.00 GB@32K  相对MHA=0.25x
同规模假想 MHA                0.500 MB/token    16.00 GB@32K  相对MHA=1.00x
同规模假想 MQA                0.016 MB/token     0.50 GB@32K  相对MHA=0.03x
Llama-3-70B 量级           0.312 MB/token    10.00 GB@32K  相对MHA=0.12x
GQA n_kv=8, FP8: 64.0 KB/token, 32K 上下文 2.00 GB
